# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

## 1. My rule and its reason codes

In [20]:
signal1 = df.copy()
signal1["staleness_bucket"] = pd.cut(
    signal1["days_since_last_update"],
    bins=[-1, 90, 180, 365, 10000],
    labels=["1_fresh_0-90d", "2_aging_90-180d", "3_stale_180-365d", "4_very_stale_365d+"]
)

staleness_table = signal1.groupby("staleness_bucket").agg(
    n=("content_id", "count"),
    pct_declining=("trend_direction", lambda x: (x == "down").mean() * 100)
).reset_index()

staleness_table

,staleness_bucket,n,pct_declining
0,1_fresh_0-90d,20655,51.203099
1,2_aging_90-180d,9171,61.105659
2,3_stale_180-365d,169,46.745562
3,4_very_stale_365d+,5,60.000000


In [17]:
signal2 = df.copy()
signal2["position_bucket"] = pd.cut(
    signal2["avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["1_top3", "2_top10", "3_top20", "4_below20"]
)

position_table = signal2.groupby("position_bucket").agg(
    n=("content_id", "count"),
    avg_ctr=("ctr", "mean")
).reset_index()

position_table

,position_bucket,n,avg_ctr
0,1_top3,1141,2.714303
1,2_top10,11842,0.651045
2,3_top20,7273,0.323443
3,4_below20,8539,0.211333


**Signal 1 — Staleness (`days_since_last_update`)** — behind FlyRank's refresh flags.

| Bucket | n | % declining |
|---|---:|---:|
| Fresh (0-90d) | 20,655 | 51.2% |
| Aging (90-180d) | 9,171 | 61.1% |
| Stale (180-365d) | 169 | 46.7% |
| Very stale (365d+) | 5 | 60.0% |

**Verdict: MIXED.** I expected decline rate to climb steadily with staleness, but the pattern isn't 
clean — "aging" pages decline the most (61.1%), while "stale" pages actually decline *less* (46.7%). 
The very-stale bucket has only 5 rows, too small to trust. Staleness alone isn't a reliable standalone 
signal here — it needs to be combined with something else (like demand/impressions) rather than used 
on its own.

**Signal 2 — CTR vs Position** — behind FlyRank's CTR-fix logic.

| Bucket | n | avg CTR |
|---|---:|---:|
| Top 3 | 1,141 | 2.71 |
| Top 10 | 11,842 | 0.65 |
| Top 20 | 7,273 | 0.32 |
| Below 20 | 8,539 | 0.21 |

**Verdict: CONFIRMED.** CTR drops cleanly and consistently as position gets worse — this is a real, 
trustworthy relationship in this data, and a solid signal to build a rule on.

**My rule:** Since staleness alone was MIXED, I won't lean on it as a strong standalone signal. 
Instead, my baseline rule combines **CTR-vs-position gap** (confirmed signal) with **demand 
(impressions_90d)**, to flag pages that get real search visibility but are under-capturing clicks 
relative to their position — a genuine "quick win" opportunity, not just a stale-page guess.

**Score formula:** opportunity_score = 0.6 * (expected_ctr_for_position - actual_ctr) + 0.4 * normalized(impressions_90d)          

**Reason code:** `ctr_below_position_expectation`

**Action label:** `"review_metadata"` (rewrite title/meta to close the CTR gap)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [18]:
import numpy as np
import os

scored = df.copy()

# Expected CTR by position bucket (from signal 2 analysis)
position_bucket_map = pd.cut(
    scored["avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["1_top3", "2_top10", "3_top20", "4_below20"]
)
expected_ctr_lookup = scored.groupby(position_bucket_map)["ctr"].transform("mean")

scored["ctr_gap"] = expected_ctr_lookup - scored["ctr"]
scored["ctr_gap"] = scored["ctr_gap"].clip(lower=0)  # only count under-performance, not over-performance

# Normalize impressions_90d to 0-1 range
scored["impressions_norm"] = (
    (scored["impressions_90d"] - scored["impressions_90d"].min()) /
    (scored["impressions_90d"].max() - scored["impressions_90d"].min())
)

# Normalize ctr_gap to 0-1 range too, so both terms are comparable
scored["ctr_gap_norm"] = (
    (scored["ctr_gap"] - scored["ctr_gap"].min()) /
    (scored["ctr_gap"].max() - scored["ctr_gap"].min())
)

scored["opportunity_score"] = (
    0.6 * scored["ctr_gap_norm"] + 0.4 * scored["impressions_norm"]
)

scored["reason_code"] = "ctr_below_position_expectation"
scored["action"] = "review_metadata"

# Only flag pages with real demand (avoid noise from near-zero impressions)
scored = scored[scored["impressions_90d"] >= 100].copy()

ranked_queue = scored.sort_values("opportunity_score", ascending=False).reset_index(drop=True)

# Write CSV
os.makedirs("../outputs", exist_ok=True)
ranked_queue[[
    "content_id", "client_id", "avg_position", "ctr", "ctr_gap",
    "impressions_90d", "opportunity_score", "reason_code", "action"
]].to_csv("../outputs/baseline_action_score.csv", index=False)

print("Rows written:", len(ranked_queue))
ranked_queue[[
    "content_id", "avg_position", "ctr", "ctr_gap", "impressions_90d", "opportunity_score"
]].head(10)

Rows written: 22006


,content_id,avg_position,ctr,ctr_gap,impressions_90d,opportunity_score
0,content_8c19996aa890,2.5,0.15,2.564303,509252,0.960304
1,content_4c36c775b818,2.3,0.41,2.304303,463103,0.867174
2,content_8451fc6f034d,2.3,0.03,2.684303,272144,0.803634
3,content_e12868d1f396,2.9,0.07,2.644303,149712,0.700197
4,content_44e481c8f55b,1.4,0.65,2.064303,312694,0.697912
5,content_4a6607efcb46,2.2,0.01,2.704303,128068,0.696738
6,content_9532f197bbc8,2.0,0.87,1.844303,309192,0.646575
7,content_8053a66bd6ac,2.6,0.08,2.634303,52687,0.623023
8,content_6f81ccd92b64,2.9,0.19,2.524303,73675,0.614923
9,content_d225ec9f3d46,0.7,0.05,2.664303,26470,0.609398


## 3. Top-20 review

In [19]:
top20 = ranked_queue.head(20)[[
    "content_id", "avg_position", "ctr", "ctr_gap", "impressions_90d", "opportunity_score"
]]
top20

,content_id,avg_position,ctr,ctr_gap,impressions_90d,opportunity_score
0,content_8c19996aa890,2.5,0.15,2.564303,509252,0.960304
1,content_4c36c775b818,2.3,0.41,2.304303,463103,0.867174
2,content_8451fc6f034d,2.3,0.03,2.684303,272144,0.803634
3,content_e12868d1f396,2.9,0.07,2.644303,149712,0.700197
4,content_44e481c8f55b,1.4,0.65,2.064303,312694,0.697912
5,content_4a6607efcb46,2.2,0.01,2.704303,128068,0.696738
6,content_9532f197bbc8,2.0,0.87,1.844303,309192,0.646575
7,content_8053a66bd6ac,2.6,0.08,2.634303,52687,0.623023
8,content_6f81ccd92b64,2.9,0.19,2.524303,73675,0.614923
9,content_d225ec9f3d46,0.7,0.05,2.664303,26470,0.609398


For all 20 rows, the action is `review_metadata` (rewrite title/meta to close the CTR gap), flagged 
by reason code `ctr_below_position_expectation`. Each row shows a page ranking well in search but 
under-capturing clicks relative to its position's expected CTR.

1. **content_8c19996aa890** — position 2.5, CTR 0.15 (gap 2.56), 509,252 impressions. Massive 
   visibility, barely any clicks. *Wrong if:* the title already matches intent well and something 
   else (e.g. a competing SERP feature) is absorbing clicks — a metadata rewrite wouldn't fix that.
2. **content_4c36c775b818** — position 2.3, CTR 0.41 (gap 2.30), 463,103 impressions. *Wrong if:* a 
   recent metadata change hasn't shown up in CTR yet (reporting lag).
3. **content_8451fc6f034d** — position 2.3, CTR 0.03 (gap 2.68), 272,144 impressions. Near-zero 
   clicks despite a great position. *Wrong if:* this is a featured-snippet/knowledge-panel page 
   where users get their answer without clicking — low CTR would reflect success, not failure.
4. **content_e12868d1f396** — position 2.9, CTR 0.07 (gap 2.64), 149,712 impressions. *Wrong if:* 
   the query mix behind these impressions is mostly navigational (users searching for something 
   else that happens to surface this page), not genuine intent-matched traffic.
5. **content_44e481c8f55b** — position 1.4, CTR 0.65 (gap 2.06), 312,694 impressions. Best position 
   in the list, still a real gap. *Wrong if:* 0.65 CTR is already reasonable for this query category 
   and my expected-CTR baseline (from the position-bucket average) is too optimistic for this case.
6. **content_4a6607efcb46** — position 2.2, CTR 0.01 (gap 2.70), 128,068 impressions. *Wrong if:* the 
   page technically ranks but is broken/unreachable (e.g. redirect issue) — a metadata fix wouldn't 
   help a page users can't actually open.
7. **content_9532f197bbc8** — position 2.0, CTR 0.87 (gap 1.84), 309,192 impressions. Already decent 
   CTR relative to others here. *Wrong if:* this page is actually performing fine and only ranks 
   high on my list because of its large impression volume, not a real metadata problem.
8. **content_8053a66bd6ac** — position 2.6, CTR 0.08 (gap 2.63), 52,687 impressions. *Wrong if:* 
   impressions are concentrated in one unusual spike day rather than steady demand, making the 
   90-day average misleading.
9. **content_6f81ccd92b64** — position 2.9, CTR 0.19 (gap 2.52), 73,675 impressions. *Wrong if:* the 
   page recently changed topic/URL and older impressions belong to a different intent than what's 
   live now.
10. **content_d225ec9f3d46** — position 0.7, CTR 0.05 (gap 2.66), 26,470 impressions. Position under 
    1 is unusual (possibly a rich result/position-zero feature). *Wrong if:* this is a featured 
    snippet where low click-through is expected by design.
11. **content_0022a6b4290f** — position 1.2, CTR 0.07 (gap 2.64), 29,747 impressions. *Wrong if:* 
    similar to above, a very strong position with structurally low CTR (e.g. answer box) isn't 
    actually a metadata problem.
12. **content_cbdf5a78dcd0** — position 2.4, CTR 0.02 (gap 2.69), 14,830 impressions. Lower volume 
    than top rows. *Wrong if:* at this impression level, the CTR estimate is noisy (few clicks total 
    can swing CTR wildly) rather than a stable signal.
13. **content_f4e210ee0c27** — position 1.6, CTR 0.06 (gap 2.65), 24,784 impressions. *Wrong if:* 
    same low-volume noise concern — small click counts make CTR unstable.
14. **content_b7bd590fe572** — position 2.9, CTR 0.02 (gap 2.69), 12,331 impressions. *Wrong if:* 
    volume here (12k impressions) is borderline low for a confident CTR read.
15. **content_998f6f88784c** — position 2.6, CTR 0.02 (gap 2.69), 12,053 impressions. *Wrong if:* 
    same low-volume caution as above.
16. **content_11900bd7941a** — position 2.8, CTR 0.41 (gap 2.30), 123,561 impressions. *Wrong if:* 
    0.41 CTR might already be acceptable for this content/intent type, and my flat expected-CTR 
    baseline doesn't account for intent differences.
17. **content_50dfd64f9e8e** — position 2.4, CTR 0.00 (gap 2.71), 4,446 impressions. Lowest volume in 
    the top 20. *Wrong if:* 4,446 impressions is too small a sample to trust a 0% CTR as a real 
    pattern rather than noise.
18. **content_134631e65b9e** — position 1.5, CTR 0.00 (gap 2.71), 4,417 impressions. *Wrong if:* same 
    low-volume noise concern as #17.
19. **content_954cc45bd437** — position 1.5, CTR 0.04 (gap 2.67), 15,439 impressions. *Wrong if:* 
    volume is on the low side for a fully confident CTR read.
20. **content_7a6df559322d** — position 0.7, CTR 0.14 (gap 2.57), 43,650 impressions. Position under 
    1 again suggests a possible rich-result page. *Wrong if:* this is another SERP-feature case where 
    low CTR is structural, not a content problem.

**Overall pattern I notice:** rows 12–19 (lower impressions, roughly 4k–25k) are exactly where my 
`impressions_norm` term should have pulled them down, but they're still ranking in the top 20 because 
their `ctr_gap` is near-maximal (close to 2.71–2.71, the ceiling). This suggests my 0.6/0.4 weighting 
may over-favor gap size vs. actual demand — worth revisiting for the model stage.

## 4. Weak picks + leakage check

**Weakest picks:** rows 12–19 stand out as the shakiest — they have low impression volume (4k–25k, 
vs. 100k+ for the top rows), meaning their CTR is based on relatively few clicks and could easily be 
noisy rather than a real, stable pattern. Rows 10, 17, and 18 also show position values under 1.5 
with CTR near or at 0.00 — this pattern looks more like a SERP-feature page (featured snippet, 
knowledge panel) than a genuine "bad metadata" case, since those formats are designed to answer the 
query without a click.

**Leakage check:** 
- No product-computed flags (`health_score`, `priority_score`, `action_type`) were used anywhere in 
  the score — only `avg_position`, `ctr`, and `impressions_90d`, all observed signals.
- No future-window data was used — `impressions_90d`, `ctr`, and `avg_position` all come from the 
  same 90-day observation window used throughout this notebook; nothing from `impressions_last_30d` 
  vs `impressions_prev_30d` (which would imply a forward-looking split) was used in this baseline.
- `trend_direction` (my w03 proxy label) was NOT used as a feature or scoring input — it only 
  appeared earlier for the staleness signal check, kept separate from the rule itself.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.